## 1.Load data

In [1]:
import pandas as pd
df = pd.read_csv("pet_stress_survey.csv")

## 2.Inspect data

In [2]:
df.head()
df.info()
df.isnull().sum()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 139 entries, 0 to 138
Data columns (total 35 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   Pet_type                      139 non-null    object
 1   Perceived_role_of_the_pet     139 non-null    object
 2   Interaction_frequency         139 non-null    object
 3   Daily_pet_care_time_minutes   139 non-null    object
 4    Pet_friendly_environment     139 non-null    object
 5   CCAS_Q1                       139 non-null    int64 
 6   CCAS_Q2                       139 non-null    int64 
 7   CCAS_Q3                       139 non-null    int64 
 8   CCAS_Q4                       139 non-null    int64 
 9   CCAS_Q5                       139 non-null    int64 
 10  CCAS_Q6                       139 non-null    int64 
 11  CCAS_Q7                       139 non-null    int64 
 12  CCAS_Q8                       139 non-null    int64 
 13  CCAS_Q9             

Pet_type                        0
Perceived_role_of_the_pet       0
Interaction_frequency           0
Daily_pet_care_time_minutes     0
 Pet_friendly_environment       0
CCAS_Q1                         0
CCAS_Q2                         0
CCAS_Q3                         0
CCAS_Q4                         0
CCAS_Q5                         0
CCAS_Q6                         0
CCAS_Q7                         0
CCAS_Q8                         0
CCAS_Q9                         0
CCAS_Q10                        0
CCAS_Q11                        0
CCAS_Q12                        0
CCAS_Q13                        0
PSS_Q1                          0
PSS_Q                           0
PSS_Q3                          0
PSS_Q4                          0
PSS_Q5                          0
PSS_Q6                          0
PSS_Q7                          0
PSS_Q8                          0
PSS_Q9                          0
PSS_Q10                         0
Owner_age                       0
Owner_gender  

## 3.Clean data

In [3]:
# Remove duplicates and Handle missing values
df = df.drop_duplicates()
df = df.dropna()
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

## 4.Encode stress levels

In [4]:
# match cleaned column names
# here we assume the PSS questions are named as follows coz before code we used .str.lower method to prevent sensitivity of column names:
reverse_pss = ["pss_q4", "pss_q5", "pss_q7", "pss_q8"]

for col in reverse_pss:
    df[col] = 4 - df[col]


In [5]:
pss_cols = [col for col in df.columns if col.startswith("pss_q")]
df["pss_score"] = df[pss_cols].sum(axis=1)


In [6]:
 df["stress_level"] = pd.cut(
    df["pss_score"],
    bins=[-1, 13, 26, 40],
    labels=["Low", "Moderate", "High"]
)


In [7]:
reverse_ccas = ["ccas_q6", "ccas_q9"]

for col in reverse_ccas:
    df[col] = 4 - df[col]


In [8]:
ccas_cols = [col for col in df.columns if col.startswith("ccas_q")]
df["ccas_score"] = df[ccas_cols].sum(axis=1)

In [9]:
df["comfort_level"] = pd.cut(
    df["ccas_score"],
    bins=[12, 25, 38, 52],
    labels=["Low Attachment", "Moderate Attachment", "High Attachment"]
)


In [10]:
df["daily_pet_care_time_minutes"] = pd.Categorical(
    df["daily_pet_care_time_minutes"],
    categories=["0-15", "16-30", "31-60", "61-120", "121-180", ">180"],
    ordered=True
)

In [11]:
df.info()
df.isna().sum()
df.head()


<class 'pandas.core.frame.DataFrame'>
Index: 136 entries, 0 to 135
Data columns (total 39 columns):
 #   Column                       Non-Null Count  Dtype   
---  ------                       --------------  -----   
 0   pet_type                     136 non-null    object  
 1   perceived_role_of_the_pet    136 non-null    object  
 2   interaction_frequency        136 non-null    object  
 3   daily_pet_care_time_minutes  136 non-null    category
 4   pet_friendly_environment     136 non-null    object  
 5   ccas_q1                      136 non-null    int64   
 6   ccas_q2                      136 non-null    int64   
 7   ccas_q3                      136 non-null    int64   
 8   ccas_q4                      136 non-null    int64   
 9   ccas_q5                      136 non-null    int64   
 10  ccas_q6                      136 non-null    int64   
 11  ccas_q7                      136 non-null    int64   
 12  ccas_q8                      136 non-null    int64   
 13  ccas_q9   

,pet_type,perceived_role_of_the_pet,interaction_frequency,daily_pet_care_time_minutes,pet_friendly_environment,ccas_q1,ccas_q2,ccas_q3,ccas_q4,ccas_q5,...,owner_gender,owner_living_situation,owner_employment_status,owner_martial_status,owner_family_income,western_province,pss_score,stress_level,ccas_score,comfort_level
0,Dog,Family member,Daily,0-15,Yes,4,4,4,4,4,...,Female,Living with partner,Student,unmarried,"350,000 -700,000",Yes,19,Moderate,46,High Attachment
1,Cat,Friend,Daily,0-15,Yes,4,4,2,2,2,...,Female,Living with family,Student,unmarried,"50,000 -150,000",Yes,22,Moderate,39,High Attachment
2,Cat,Family member,Daily,16-30,Yes,3,3,3,3,3,...,Female,Living with family,Student,unmarried,"150,000 -350,000",Yes,29,High,37,Moderate Attachment
3,Dog,Friend,Daily,61-120,Yes,3,3,2,4,3,...,Male,Living with family,Student,unmarried,"Below 50,000",Yes,20,Moderate,39,High Attachment
4,Cat,Family member,Daily,31-60,Yes,3,3,2,3,2,...,Female,Living with family,Student,unmarried,"150,000 -350,000",Yes,24,Moderate,34,Moderate Attachment


## 5.Exploratory Analysis

In [12]:
df["pss_score"].mean()
df.groupby("pet_type")["pss_score"].mean()
df.corr(numeric_only=True)


,ccas_q1,ccas_q2,ccas_q3,ccas_q4,ccas_q5,ccas_q6,ccas_q7,ccas_q8,ccas_q9,ccas_q10,...,pss_q4,pss_q5,pss_q6,pss_q7,pss_q8,pss_q9,pss_q10,owner_age,pss_score,ccas_score
ccas_q1,1.000000,0.593851,0.523747,0.478812,0.503157,-0.460392,0.468668,0.556718,-0.342037,0.475003,...,0.069028,0.197490,-0.148410,0.026985,0.129677,-0.097886,-0.110101,-0.012613,-0.020500,0.713034
ccas_q2,0.593851,1.000000,0.494288,0.522141,0.595512,-0.573778,0.528465,0.621256,-0.356520,0.482691,...,0.047765,0.188648,-0.105849,0.083500,0.077132,-0.081313,-0.002786,-0.058434,0.045147,0.743633
ccas_q3,0.523747,0.494288,1.000000,0.480772,0.545997,-0.509981,0.594431,0.626995,-0.432985,0.476308,...,0.091308,0.203741,-0.127682,0.062444,0.105886,-0.054355,-0.121025,-0.019207,-0.032272,0.752816
ccas_q4,0.478812,0.522141,0.480772,1.000000,0.535995,-0.621667,0.614346,0.612377,-0.493426,0.583226,...,0.092647,0.193823,-0.084118,0.093729,0.054859,-0.041626,-0.060216,-0.067965,-0.072729,0.734816
ccas_q5,0.503157,0.595512,0.545997,0.535995,1.000000,-0.675305,0.587685,0.547299,-0.495227,0.495354,...,0.062719,0.138210,-0.107683,0.153007,0.066432,-0.014397,0.010170,-0.141100,0.004989,0.712940
ccas_q6,-0.460392,-0.573778,-0.509981,-0.621667,-0.675305,1.000000,-0.452744,-0.579059,0.536258,-0.557627,...,-0.117406,-0.288849,0.112268,-0.149160,-0.195554,0.060737,0.031394,0.032850,-0.034630,-0.592201
ccas_q7,0.468668,0.528465,0.594431,0.614346,0.587685,-0.452744,1.000000,0.690143,-0.529596,0.506304,...,-0.057118,0.086036,0.028471,0.045198,-0.029691,0.133757,0.097841,-0.020638,0.042975,0.821947
ccas_q8,0.556718,0.621256,0.626995,0.612377,0.547299,-0.579059,0.690143,1.000000,-0.487221,0.564228,...,-0.013364,0.148293,0.002772,0.028495,-0.016794,0.030772,0.064078,-0.021663,0.002478,0.834324
ccas_q9,-0.342037,-0.356520,-0.432985,-0.493426,-0.495227,0.536258,-0.529596,-0.487221,1.000000,-0.374687,...,-0.116646,-0.192633,0.120409,-0.102058,-0.128159,0.116519,0.057977,0.003363,0.093710,-0.466224
ccas_q10,0.475003,0.482691,0.476308,0.583226,0.495354,-0.557627,0.506304,0.564228,-0.374687,1.000000,...,0.015654,0.097903,-0.040550,0.094274,0.058716,0.010404,-0.060605,-0.141847,-0.018855,0.696554


## 6.Save cleaned dataset

In [14]:
df.to_csv("pet_stress_processed.csv", index=False)
